# GPU inference-service validation
Select an NVIDIA T4 runtime, then run the cells in order. The notebook downloads the project directly from GitHub; no source ZIP, Drive mount, or paid API is needed.

The default short group runs 30 requests with no-cache/LRU/learned and two copy modes: 5 service runs / 150 requests in float32. It downloads SmolLM2-135M. Timings include instrumentation; no speedup is assumed. Download results immediately because Colab files are temporary.


In [ ]:
import os, sys, subprocess, hashlib, json
from pathlib import Path
from google.colab import files
repo = Path("/content/kv-serving-source")
url = "https://github.com/swathiblrs/Learning-Based-KV-Cache-Management-for-GPU-LLM-Inference.git"
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, str(repo)], check=True)
os.chdir(repo)
checksums = {str(p.relative_to(repo)): hashlib.sha256(p.read_bytes()).hexdigest()
             for p in (repo / "src").rglob("*.py")}
(repo / "source-manifest.json").write_text(json.dumps(checksums, indent=2))
os.environ["USE_TF"] = "0"
subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv>=20.26,<21"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages", "--no-download", ".colab-venv"], check=True)
python = str(repo / ".colab-venv/bin/python")
subprocess.run([python, "-m", "pip", "install", "-e", ".[serving]"], check=True)


In [ ]:
subprocess.run([python, "-c", "import torch; assert torch.cuda.is_available(), 'Select GPU runtime'; print(torch.cuda.get_device_name())"], check=True)
subprocess.run([python, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
subprocess.run([python, "-m", "kv_cache.cli", "experiment", "--output", "results/experiment"], check=True)


## Run one group
Keep the defaults for the first run. After it passes, change CONCURRENCY to 4 for queueing validation, or WORKLOAD to shift/scan. Increase repeats and use additional held-out seeds for performance evidence. Each run creates a new directory. Servers listen only on loopback and stop automatically. The ZIP is refreshed after every completed configuration.


In [ ]:
CONCURRENCY = 1
WORKLOAD = "hot"
SEED = 99
REQUESTS = 30
REPEATS = 1
BUDGET_MIB = 32
from datetime import datetime, timezone
run_dir = Path("/content") / ("kv-serving-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f"))
archive = str(run_dir) + ".zip"
command = [python, "-u", "-m", "kv_cache.serve_eval", "--predictor", "results/experiment/predictor.json",
           "--device", "cuda", "--requests", str(REQUESTS), "--repeats", str(REPEATS),
           "--seed", str(SEED), "--workload", WORKLOAD, "--budget-mib", str(BUDGET_MIB),
           "--concurrency", str(CONCURRENCY), "--output", str(run_dir)]
try:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("Evaluation failed; download diagnostics below")
finally:
    if run_dir.exists():
        import shutil
        shutil.copyfile(repo / "source-manifest.json", run_dir / "source-manifest.json")
        shutil.make_archive(str(run_dir), "zip", run_dir)
    if Path(archive).exists():
        files.download(archive)


In [ ]:
manifest = json.loads((run_dir / "manifest.json").read_text())
print("Status:", manifest["status"])
for row in manifest["completed"]:
    print(row["label"], row["correctness"]["status"],
          "hit rate:", round(row["hit_rate"], 3),
          "HTTP p95 ms:", round(row["http_p95_ms"], 2),
          "requests/s:", round(row["requests_per_second"], 2))
print("Do not claim a speedup from one short synthetic run.")


## Download again if needed
This cell only downloads the most recent ZIP. It does not rerun the experiment. If the runtime was reset, an undownloaded archive may no longer exist.


In [ ]:
from pathlib import Path
from google.colab import files
archives = sorted(Path("/content").glob("kv-serving-*.zip"))
if archives:
    files.download(str(archives[-1]))
else:
    print("No serving archive found in this runtime.")
